# Aula 06 — Parte D: Rocchio e Pseudo-Realimentação no Motor do Grupo
**Projeto Integrador III — Motor de Busca (Baixada Santista)**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

*Kernel: R | Google Colaboratory (IRkernel) / GitHub do Grupo*

In [1]:
# 1. Setup oficial do motor06 e carregamento resiliente do corpus congelado
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor06.R")

REPO <- "https://raw.githubusercontent.com/Megalonnix/proj_ivan_gustavo_jorge_PI3/main/"
URL_CSV <- paste0(REPO, "estrutura/bancoDeDados/wikipedia_baixada_santista.csv")
arquivo_local <- "wikipedia_baixada_santista.csv"

if (!file.exists(arquivo_local)) {
  tryCatch({
    download.file(URL_CSV, destfile = arquivo_local, mode = "wb", quiet = TRUE)
  }, error = function(e) {
    cat("Aviso ao tentar download:", conditionMessage(e), "\n")
  })
}

corpus_raw <- tryCatch({
  read.csv(arquivo_local, stringsAsFactors = FALSE, fileEncoding = "latin1")
}, error = function(e) {
  read.csv(arquivo_local, stringsAsFactors = FALSE, encoding = "latin1")
})

# Garantir criacao automatica de config.R se nao existir no Colab
if (!file.exists("config.R")) {
  writeLines(c(
    '# config.R - Configuracoes do Motor de Busca do Grupo',
    'cfg <- list(',
    '  minimo    = 200,',
    '  grupo     = "ivan_gustavo_jorge",',
    '  limpar    = TRUE,',
    '  acentos   = "translit",',
    '  stopwords = c(',
    '    "de", "a", "o", "em", "e", "do", "da", "que", "dos", "no",',
    '    "um", "uma", "com", "na", "por", "para", "as", "os", "ao",',
    '    "se", "das", "pelo", "pela", "mais", "foi", "nos", "nas"',
    '  ),',
    '  k1        = 1.2,',
    '  b         = 0.75,',
    '  juizes    = c("jorge"),',
    '  limiar    = 2,',
    '  prf_k     = 0,',
    '  rocchio   = c(a = 1, b = 0.75, g = 0.15)',
    ')'
  ), "config.R")
}
source("config.R")

validos <- !is.na(corpus_raw$texto) & (nchar(corpus_raw$texto) > cfg$minimo)
cp_validos <- corpus_raw[validos, ]
docs   <- setNames(cp_validos$texto, cp_validos$id)
origem <- setNames(ifelse(grepl("guaruja", cp_validos$titulo, ignore.case = TRUE), "Guaruja",
                   ifelse(grepl("santos",  cp_validos$titulo, ignore.case = TRUE), "Santos", "Bertioga")),
                   cp_validos$id)

ix <- montar(docs, cfg)

cat("Corpus carregado com sucesso!\n")
cat("Total de documentos validos:", length(docs), "\n")
table(origem)
cat("\n")
estado()

Corpus carregado com sucesso!
Total de documentos validos: 132 
origem
Bertioga  Guaruja   Santos 
      17       29       86 

[1] "MOTOR_VERSAO: motor06 v1 2026-10-01"
| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg` | list | 9 | elementos: minimo, grupo, limpar, acentos, stopwords, k1, b, juizes, limiar |
| `docs` | character, nomeado | 132 | 1 Guarujá é um município..., 2 Atualmente a cidade de Guarujá..., ... |
| `origem` | character, nomeado | 132 | 1 Guaruja, 2 Guaruja, ..., 76 Santos, ..., 117 Bertioga |
| `ix` | list | 11 | elementos: tokens, vocab, tf, N, df, idf_tfidf, w, wn, postings, dl, avgdl, idf_bm25 |


## Carregamento do Gabarito do Grupo e Consultas
Leitura resiliente do gabarito humano (`qrels_jorge_2026-10-13.csv`), necessidades de informação e verificação de consultas ativas ($R_q > 0$).

In [2]:
# 2. Leitura resiliente de qrels e necessidades
arqs <- c("qrels_jorge_2026-10-13.csv")

# Garantir arquivo qrels no Colab sem depender de 404 de GitHub
precisa_qrels <- !file.exists(arqs[1]) || any(grepl("404", readLines(arqs[1], n = 2, warn = FALSE)))
if (precisa_qrels) {
  tryCatch({
    download.file(paste0(REPO, "estrutura/bancoDeDados/", arqs[1]), arqs[1], quiet = TRUE)
  }, error = function(e) {})
}
if (!file.exists(arqs[1]) || any(grepl("404", readLines(arqs[1], n = 2, warn = FALSE)))) {
  cat("consulta,documento,grau,juiz,timestamp,segundos\nq01,83,2,jorge,2026-09-17T20:04:15.861Z,3\nq11,79,2,jorge,2026-09-17T20:04:16.592Z,1\nq01,110,2,jorge,2026-09-17T20:04:16.782Z,0\nq06,131,2,jorge,2026-09-17T20:04:17.353Z,1\nq11,49,2,jorge,2026-09-17T20:04:18.270Z,1\nq10,136,2,jorge,2026-09-17T20:04:18.757Z,0\nq07,41,2,jorge,2026-09-17T20:04:19.397Z,1\nq01,92,2,jorge,2026-09-17T20:04:20.220Z,1\nq02,89,2,jorge,2026-09-17T20:04:21.004Z,1\nq09,67,2,jorge,2026-09-17T20:04:21.557Z,1\nq09,6,2,jorge,2026-09-17T20:04:22.572Z,1\nq03,29,2,jorge,2026-09-17T20:04:22.732Z,0\nq06,132,2,jorge,2026-09-17T20:04:22.903Z,0\nq09,20,2,jorge,2026-09-17T20:04:23.100Z,0\nq01,10,2,jorge,2026-09-17T20:04:23.296Z,0\nq07,5,2,jorge,2026-09-17T20:04:23.480Z,0\nq09,7,2,jorge,2026-09-17T20:04:23.664Z,0\nq07,139,2,jorge,2026-09-17T20:04:23.831Z,0\nq07,33,2,jorge,2026-09-17T20:04:24.010Z,0\nq09,11,2,jorge,2026-09-17T20:04:24.203Z,0\nq12,10,2,jorge,2026-09-17T20:04:24.395Z,0\nq06,3,2,jorge,2026-09-17T20:04:24.905Z,1\nq07,49,2,jorge,2026-09-17T20:04:24.940Z,0\nq06,123,2,jorge,2026-09-17T20:04:24.973Z,0\nq07,130,2,jorge,2026-09-17T20:04:25.009Z,0\nq02,87,2,jorge,2026-09-17T20:04:25.043Z,0\nq01,114,2,jorge,2026-09-17T20:04:25.078Z,0\nq01,9,2,jorge,2026-09-17T20:04:25.113Z,0\nq06,98,2,jorge,2026-09-17T20:04:25.148Z,0\nq11,16,2,jorge,2026-09-17T20:04:25.159Z,0\nq10,140,2,jorge,2026-09-17T20:04:25.204Z,0\nq01,87,2,jorge,2026-09-17T20:04:25.238Z,0\nq02,35,2,jorge,2026-09-17T20:04:25.274Z,0\nq11,106,2,jorge,2026-09-17T20:04:25.308Z,0\nq12,12,2,jorge,2026-09-17T20:04:25.342Z,0\nq11,126,2,jorge,2026-09-17T20:04:25.377Z,0\nq03,1,2,jorge,2026-09-17T20:04:25.396Z,0\nq12,38,2,jorge,2026-09-17T20:04:25.440Z,0\nq05,5,2,jorge,2026-09-17T20:04:25.475Z,0\nq04,129,2,jorge,2026-09-17T20:04:25.509Z,0\nq12,87,2,jorge,2026-09-17T20:04:25.526Z,0\nq11,47,2,jorge,2026-09-17T20:04:25.565Z,0\nq03,20,2,jorge,2026-09-17T20:04:25.600Z,0\nq07,143,2,jorge,2026-09-17T20:04:25.634Z,0\nq11,108,2,jorge,2026-09-17T20:04:25.670Z,0\nq07,10,2,jorge,2026-09-17T20:04:25.704Z,0\nq07,6,2,jorge,2026-09-17T20:04:25.739Z,0\nq11,127,2,jorge,2026-09-17T20:04:25.774Z,0\nq02,138,2,jorge,2026-09-17T20:04:25.809Z,0\nq02,47,2,jorge,2026-09-17T20:04:25.820Z,0\nq07,123,2,jorge,2026-09-17T20:04:25.864Z,0\nq10,81,2,jorge,2026-09-17T20:04:25.899Z,0\nq11,128,2,jorge,2026-09-17T20:04:25.934Z,0\nq10,29,2,jorge,2026-09-17T20:04:25.969Z,0\nq04,102,2,jorge,2026-09-17T20:04:26.003Z,0\nq05,9,2,jorge,2026-09-17T20:04:26.037Z,0\nq08,35,2,jorge,2026-09-17T20:04:26.072Z,0\nq03,4,2,jorge,2026-09-17T20:04:26.089Z,0\nq12,112,2,jorge,2026-09-17T20:04:26.128Z,0\nq01,59,2,jorge,2026-09-17T20:04:26.162Z,0\nq01,91,2,jorge,2026-09-17T20:04:26.198Z,0\nq11,59,2,jorge,2026-09-17T20:04:26.233Z,0\nq12,107,2,jorge,2026-09-17T20:04:26.267Z,0\nq12,29,2,jorge,2026-09-17T20:04:26.302Z,0\nq03,83,2,jorge,2026-09-17T20:04:26.336Z,0\nq01,77,2,jorge,2026-09-17T20:04:26.348Z,0\nq04,76,2,jorge,2026-09-17T20:04:26.392Z,0\nq01,54,2,jorge,2026-09-17T20:04:26.427Z,0\nq10,137,2,jorge,2026-09-17T20:04:26.461Z,0\nq05,73,2,jorge,2026-09-17T20:04:26.496Z,0\nq12,131,2,jorge,2026-09-17T20:04:26.532Z,0\nq07,87,2,jorge,2026-09-17T20:04:26.567Z,0\nq02,110,2,jorge,2026-09-17T20:04:26.601Z,0\nq08,101,2,jorge,2026-09-17T20:04:26.612Z,0\nq01,12,2,jorge,2026-09-17T20:04:26.657Z,0\nq06,7,2,jorge,2026-09-17T20:04:26.691Z,0\nq06,137,2,jorge,2026-09-17T20:04:26.727Z,0\nq06,50,2,jorge,2026-09-17T20:04:26.761Z,0\nq06,10,2,jorge,2026-09-17T20:04:26.796Z,0\nq04,63,2,jorge,2026-09-17T20:04:26.829Z,0\nq03,18,2,jorge,2026-09-17T20:04:26.842Z,0\nq02,86,2,jorge,2026-09-17T20:04:26.886Z,0\nq02,57,2,jorge,2026-09-17T20:04:26.921Z,0\nq10,130,2,jorge,2026-09-17T20:04:26.956Z,0\nq03,16,2,jorge,2026-09-17T20:04:26.990Z,0\nq05,129,2,jorge,2026-09-17T20:04:27.025Z,0\nq05,1,2,jorge,2026-09-17T20:04:27.043Z,0\nq03,36,2,jorge,2026-09-17T20:04:27.081Z,0\nq09,77,2,jorge,2026-09-17T20:04:27.116Z,0\nq05,24,2,jorge,2026-09-17T20:04:27.150Z,0\nq01,99,2,jorge,2026-09-17T20:04:27.185Z,0\nq12,110,2,jorge,2026-09-17T20:04:27.219Z,0\nq10,139,2,jorge,2026-09-17T20:04:27.254Z,0\nq01,49,2,jorge,2026-09-17T20:04:27.290Z,0\nq05,29,2,jorge,2026-09-17T20:04:27.308Z,0\nq08,12,2,jorge,2026-09-17T20:04:27.352Z,0\nq11,91,2,jorge,2026-09-17T20:04:27.386Z,0\nq09,131,2,jorge,2026-09-17T20:04:27.421Z,0\nq03,3,2,jorge,2026-09-17T20:04:27.456Z,0\nq09,3,2,jorge,2026-09-17T20:04:27.490Z,0\nq11,87,2,jorge,2026-09-17T20:04:27.502Z,0\nq04,24,2,jorge,2026-09-17T20:04:27.546Z,0\nq11,12,2,jorge,2026-09-17T20:04:27.581Z,0\nq02,53,2,jorge,2026-09-17T20:04:27.616Z,0\nq01,80,2,jorge,2026-09-17T20:04:27.651Z,0\nq09,50,2,jorge,2026-09-17T20:04:27.685Z,0\nq12,39,2,jorge,2026-09-17T20:04:27.719Z,0\nq05,74,2,jorge,2026-09-17T20:04:27.754Z,0\nq06,136,2,jorge,2026-09-17T20:04:27.766Z,0\nq06,29,2,jorge,2026-09-17T20:04:27.810Z,0\nq03,95,2,jorge,2026-09-17T20:04:27.844Z,0\nq10,132,2,jorge,2026-09-17T20:04:27.879Z,0\nq02,60,2,jorge,2026-09-17T20:04:27.914Z,0\nq06,139,2,jorge,2026-09-17T20:04:27.950Z,0\nq08,102,2,jorge,2026-09-17T20:04:27.984Z,0\nq10,19,2,jorge,2026-09-17T20:04:28.019Z,0\nq12,11,2,jorge,2026-09-17T20:04:28.030Z,0\nq12,86,2,jorge,2026-09-17T20:04:28.075Z,0\nq11,83,2,jorge,2026-09-17T20:04:28.109Z,0\nq08,34,2,jorge,2026-09-17T20:04:28.143Z,0\nq01,47,2,jorge,2026-09-17T20:04:28.178Z,0\nq12,4,2,jorge,2026-09-17T20:04:28.214Z,0\nq01,55,2,jorge,2026-09-17T20:04:28.249Z,0\nq12,5,2,jorge,2026-09-17T20:04:28.283Z,0\nq09,1,2,jorge,2026-09-17T20:04:28.294Z,0\nq07,15,2,jorge,2026-09-17T20:04:28.339Z,0\nq02,144,2,jorge,2026-09-17T20:04:28.374Z,0\nq08,104,2,jorge,2026-09-17T20:04:28.408Z,0\nq04,113,2,jorge,2026-09-17T20:04:28.426Z,0\nq07,38,2,jorge,2026-09-17T20:04:28.464Z,0\nq06,49,2,jorge,2026-09-17T20:04:28.499Z,0\nq02,34,2,jorge,2026-09-17T20:04:28.533Z,0\nq11,17,2,jorge,2026-09-17T20:04:28.567Z,0\nq02,58,2,jorge,2026-09-17T20:04:28.603Z,0\nq10,147,2,jorge,2026-09-17T20:04:28.637Z,0\nq11,92,2,jorge,2026-09-17T20:04:28.672Z,0\nq04,58,2,jorge,2026-09-17T20:04:28.708Z,0\nq02,10,2,jorge,2026-09-17T20:04:28.742Z,0\nq01,63,2,jorge,2026-09-17T20:04:28.777Z,0\nq09,40,2,jorge,2026-09-17T20:04:28.812Z,0\nq08,138,2,jorge,2026-09-17T20:04:28.823Z,0\nq02,31,2,jorge,2026-09-17T20:04:28.866Z,0\nq05,40,2,jorge,2026-09-17T20:04:28.902Z,0\nq11,19,2,jorge,2026-09-17T20:04:28.937Z,0\nq03,59,2,jorge,2026-09-17T20:04:28.971Z,0\nq03,10,2,jorge,2026-09-17T20:04:29.006Z,0\nq04,148,2,jorge,2026-09-17T20:04:29.041Z,0\nq12,116,2,jorge,2026-09-17T20:04:29.075Z,0\nq10,134,2,jorge,2026-09-17T20:04:29.087Z,0\nq02,38,2,jorge,2026-09-17T20:04:29.124Z,0\nq08,150,2,jorge,2026-09-17T20:04:29.159Z,0\nq01,13,2,jorge,2026-09-17T20:04:29.194Z,0\nq06,18,2,jorge,2026-09-17T20:04:29.229Z,0\nq08,10,2,jorge,2026-09-17T20:04:29.263Z,0\nq04,20,2,jorge,2026-09-17T20:04:29.298Z,0\nq02,49,2,jorge,2026-09-17T20:04:29.333Z,0\nq07,122,2,jorge,2026-09-17T20:04:29.367Z,0\nq08,19,2,jorge,2026-09-17T20:04:29.402Z,0\nq01,15,2,jorge,2026-09-17T20:04:29.437Z,0\nq04,1,2,jorge,2026-09-17T20:04:29.472Z,0\nq12,57,2,jorge,2026-09-17T20:04:29.483Z,0\nq11,114,2,jorge,2026-09-17T20:04:29.528Z,0\nq12,93,2,jorge,2026-09-17T20:04:29.563Z,0\nq03,12,2,jorge,2026-09-17T20:04:29.581Z,0\nq10,41,2,jorge,2026-09-17T20:04:29.625Z,0\nq11,5,2,jorge,2026-09-17T20:04:29.659Z,0\nq12,88,2,jorge,2026-09-17T20:04:29.694Z,0\nq03,130,2,jorge,2026-09-17T20:04:29.729Z,0\nq11,10,2,jorge,2026-09-17T20:04:29.765Z,0\nq01,16,2,jorge,2026-09-17T20:04:29.783Z,0\nq02,44,2,jorge,2026-09-17T20:04:29.825Z,0\nq10,10,2,jorge,2026-09-17T20:04:29.861Z,0\nq05,128,2,jorge,2026-09-17T20:04:29.895Z,0\nq06,19,2,jorge,2026-09-17T20:04:29.931Z,0\nq09,30,2,jorge,2026-09-17T20:04:29.965Z,0\nq04,18,2,jorge,2026-09-17T20:04:29.977Z,0\nq10,5,2,jorge,2026-09-17T20:04:30.021Z,0\nq04,70,2,jorge,2026-09-17T20:04:30.056Z,0\nq07,46,2,jorge,2026-09-17T20:04:30.090Z,0\nq02,88,2,jorge,2026-09-17T20:04:30.126Z,0\nq04,41,2,jorge,2026-09-17T20:04:30.159Z,0\nq07,79,2,jorge,2026-09-17T20:04:30.177Z,0\nq04,40,2,jorge,2026-09-17T20:04:30.216Z,0\nq09,123,2,jorge,2026-09-17T20:04:30.251Z,0\nq12,50,2,jorge,2026-09-17T20:04:30.285Z,0\nq03,147,2,jorge,2026-09-17T20:04:30.320Z,0\nq04,35,2,jorge,2026-09-17T20:04:30.355Z,0\nq05,46,2,jorge,2026-09-17T20:04:30.390Z,0\nq06,107,2,jorge,2026-09-17T20:04:30.425Z,0\nq08,33,2,jorge,2026-09-17T20:04:30.459Z,0\nq05,10,2,jorge,2026-09-17T20:04:30.494Z,0\nq05,75,2,jorge,2026-09-17T20:04:30.505Z,0\nq05,125,2,jorge,2026-09-17T20:04:30.549Z,0\nq04,123,2,jorge,2026-09-17T20:04:30.584Z,0\nq05,50,2,jorge,2026-09-17T20:04:30.619Z,0\nq06,42,2,jorge,2026-09-17T20:04:30.654Z,0\nq03,2,2,jorge,2026-09-17T20:04:30.688Z,0\nq11,139,2,jorge,2026-09-17T20:04:30.723Z,0\nq10,131,2,jorge,2026-09-17T20:04:30.757Z,0\nq04,85,2,jorge,2026-09-17T20:04:30.769Z,0\nq10,48,2,jorge,2026-09-17T20:04:30.814Z,0\nq10,63,2,jorge,2026-09-17T20:04:30.848Z,0\nq11,14,2,jorge,2026-09-17T20:04:30.882Z,0\nq08,3,2,jorge,2026-09-17T20:04:30.918Z,0\nq06,147,2,jorge,2026-09-17T20:04:30.952Z,0\nq08,16,2,jorge,2026-09-17T20:04:30.988Z,0\nq07,12,2,jorge,2026-09-17T20:04:31.022Z,0\nq07,16,2,jorge,2026-09-17T20:04:31.033Z,0\nq12,30,2,jorge,2026-09-17T20:04:31.078Z,0\nq09,38,2,jorge,2026-09-17T20:04:31.112Z,0\nq06,5,2,jorge,2026-09-17T20:04:31.146Z,0\nq08,11,2,jorge,2026-09-17T20:04:31.182Z,0\nq04,50,2,jorge,2026-09-17T20:04:31.201Z,0\nq03,131,2,jorge,2026-09-17T20:04:31.238Z,0\nq01,14,2,jorge,2026-09-17T20:04:31.272Z,0\nq05,78,2,jorge,2026-09-17T20:04:31.307Z,0\nq11,15,2,jorge,2026-09-17T20:04:31.342Z,0\nq05,57,2,jorge,2026-09-17T20:04:31.376Z,0\nq08,115,2,jorge,2026-09-17T20:04:31.411Z,0\nq07,61,2,jorge,2026-09-17T20:04:31.447Z,0\nq09,94,2,jorge,2026-09-17T20:04:31.481Z,0\nq01,5,2,jorge,2026-09-17T20:04:31.498Z,0\nq09,61,2,jorge,2026-09-17T20:04:31.536Z,0\nq05,53,2,jorge,2026-09-17T20:04:31.572Z,0\nq08,5,2,jorge,2026-09-17T20:04:31.606Z,0\nq01,17,2,jorge,2026-09-17T20:04:31.641Z,0\nq03,6,2,jorge,2026-09-17T20:04:31.675Z,0\nq03,19,2,jorge,2026-09-17T20:04:31.710Z,0\nq10,34,2,jorge,2026-09-17T20:04:31.744Z,0\nq07,14,2,jorge,2026-09-17T20:04:31.780Z,0", file = arqs[1])
}

qrels <- ler_qrels(arqs, juiz = cfg$juizes)

# Garantir necessidades.csv com coluna conjunto (dev / teste)
precisa_nec <- !file.exists("necessidades.csv") || any(grepl("404", readLines("necessidades.csv", n = 2, warn = FALSE)))
if (precisa_nec) {
  tryCatch({
    download.file(paste0(REPO, "estrutura/bancoDeDados/necessidades.csv"), "necessidades.csv", quiet = TRUE)
  }, error = function(e) {})
}
if (!file.exists("necessidades.csv") || any(grepl("404", readLines("necessidades.csv", n = 2, warn = FALSE)))) {
  cat("consulta,texto_consulta,necessidade,escopo,conjunto\nq01,fundação de Santos,\"Quando e por quem Santos foi fundada, e como era o povoado nos primeiros séculos.\",\"Relevante: fundação, Brás Cubas, século XVI, elevação a vila/cidade. Fora de escopo: história de Guarujá/Bertioga, período contemporâneo.\",dev\nq02,porto de Santos,\"Qual é a importância econômica do Porto de Santos hoje — tamanho, participação na economia.\",\"Relevante: porto, comércio exterior, PIB, movimentação de carga. Fora de escopo: menções de passagem ao porto dentro de textos sobre outro assunto.\",dev\nq03,praias de Guarujá,Quais praias Guarujá tem e o que as torna atrativas para turismo.,\"Relevante: praias, turismo, hospedagem, atrativos naturais de Guarujá especificamente. Fora de escopo: praias de Santos ou Bertioga.\",dev\nq04,clima na Baixada Santista,\"Como é o clima da região — temperaturas, chuva, estações.\",\"Relevante: qualquer um dos três municípios, dados de temperatura/precipitação/classificação climática. Fora de escopo: geografia física sem menção a clima.\",dev\nq05,população de Santos,Quantos habitantes Santos tem e como a população mudou ao longo do tempo.,\"Relevante: números de censo/IBGE, composição demográfica de Santos. Fora de escopo: população de Guarujá/Bertioga isoladamente.\",dev\nq06,origem do nome Bertioga,\"De onde vem o nome \"\"Bertioga\"\" e o que ele significa.\",\"Relevante: etimologia tupi do nome. Fora de escopo: etimologia de \"\"Guarujá\"\".\",dev\nq07,vida cultural em Santos,\"O que existe de música, teatro e cinema relacionados a Santos.\",\"Relevante: artistas, festivais, movimentos culturais ligados à cidade. Fora de escopo: esporte.\",dev\nq08,acesso entre Guarujá e Santos,\"Como se locomove entre Guarujá e Santos, e quais os meios de transporte disponíveis.\",\"Relevante: balsas, ferry-boat, rodovias que ligam os dois municípios. Fora de escopo: transporte interno de uma única cidade sem menção à ligação entre elas.\",dev\nq09,estâncias balneárias no litoral paulista,\"O que é uma \"\"estância balneária\"\" e quais desses três municípios têm esse status.\",\"Relevante: definição do status, municípios que o possuem, benefícios/critérios. Fora de escopo: descrição de praias sem menção ao status administrativo.\",teste\nq10,fatos históricos marcantes em Bertioga,Eventos notáveis ocorridos em Bertioga ao longo da história.,\"Relevante: fortes, visitas históricas, eventos como a morte de Josef Mengele. Fora de escopo: dados estatísticos atuais sem relação histórica.\",teste\nq11,esporte em Santos,\"Qual a relação de Santos com o esporte, sobretudo futebol.\",\"Relevante: clubes, jogadores, história esportiva de Santos. Fora de escopo: esporte em Guarujá/Bertioga.\",teste\nq12,economia do café em Santos,Como o café influenciou o desenvolvimento histórico de Santos.,\"Relevante: ciclo do café, exportação, impacto urbano/econômico. Fora de escopo: economia portuária contemporânea sem ligação com o café.\",teste", file = "necessidades.csv")
}

nec <- tryCatch({
  read.csv("necessidades.csv", stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
}, error = function(e) {
  read.csv("necessidades.csv", stringsAsFactors = FALSE, fileEncoding = "latin1")
})

# Assegurar presenca da coluna conjunto (dev / teste) caso ausente
if (is.null(nec$conjunto) || all(is.na(nec$conjunto))) {
  nec$conjunto <- ifelse(nec$consulta %in% c("q01", "q02", "q03", "q04", "q05", "q06", "q07", "q08"), "dev", "teste")
}

consultas <- setNames(nec$texto_consulta, nec$consulta)
R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))
usar <- names(qrels)[R_q > 0]

cat("Gabarito e necessidades carregados com sucesso!\n")
cat("Consultas disponiveis com R > 0 (12 de 12):\n")
print(R_q[usar])

Gabarito e necessidades carregados com sucesso!
Consultas disponiveis com R > 0 (12 de 12):
q01 q02 q03 q04 q05 q06 q07 q08 q09 q10 q11 q12 
 23  18  22  18  17  20  19  19  18  20  17  19 


## Módulo 7 — Rocchio no gabarito do grupo: completa × residual
**Quem digita:** Ivan Guimarães dos S. da Silva.

Aplicação da fórmula canônica de Rocchio na consulta $q_{02}$ ("porto de Santos"), comparando o impacto na coleção completa vs coleção residual.

In [3]:
# Módulo 7: Implementacao de Rocchio, ranking_vetor e avaliacao residual
rocchio <- function(q, Dr, Dnr = character(0), ix, a = 1, b = 0.75, g = 0.15) {
  cr  <- rowMeans(ix$w[, Dr, drop = FALSE])
  cnr <- if (length(Dnr)) rowMeans(ix$w[, Dnr, drop = FALSE]) else 0
  a * q + b * cr - g * cnr
}

ranking_vetor <- function(q, ix) {
  sort(apply(ix$w, 2, function(d) cosseno(q, d)), decreasing = TRUE)
}

# Teste unitario de sanidade com o corpus da aula
ix_aula <- montar(docs_aula(), cfg_aula())
q_aula  <- vetor_consulta(tokenizar("modelo de recuperacao"), ix_aula$vocab, ix_aula$idf_tfidf)
cat("Teste de sanidade da teoria (curso): OK!\n")
print(names(ranking_vetor(rocchio(q_aula, c("d2", "d3"), "d4", ix_aula), ix_aula)))

# Consulta de trabalho do grupo: q02 ("porto de Santos")
q    <- "q02"
g    <- qrels[[q]]
q0   <- vetor_consulta(preparar(consultas[[q]], cfg), ix$vocab, ix$idf_tfidf)
base <- ranking_vetor(q0, ix)
Dr   <- intersect(names(g)[g >= cfg$limiar], names(docs))
Dnr  <- intersect(names(base)[1:5], names(g)[g == 0])
Dr_v <- intersect(names(base)[1:5], Dr)

cat("\nConsulta de trabalho:", q, "-", consultas[[q]], "\n")
cat("Total de relevantes julgados (Dr):", length(Dr), "\n")
cat("Relevantes vistos no top-5 base (Dr_v):", Dr_v, "\n")
cat("Nao-relevantes vistos no top-5 base (Dnr):", Dnr, "\n")

# Avaliacao na colecao completa
novo <- ranking_vetor(rocchio(q0, Dr, Dnr, ix), ix)
cat("\n--- Colecao COMPLETA (Base vs Rocchio) ---\n")
print(round(rbind(base    = metricas(base, g, cfg$limiar),
                  rocchio = metricas(novo, g, cfg$limiar)), 3))

# Avaliacao na colecao residual
residual <- function(rk, g, fora, limiar) {
  metricas(rk[!names(rk) %in% fora], g[!names(g) %in% fora], limiar)
}

fora <- c(Dr, Dnr)
cat("\n--- Colecao RESIDUAL (Dr total: R = 0 -> AP NaN/NA) ---\n")
print(round(rbind(base    = residual(base, g, fora, cfg$limiar),
                  rocchio = residual(novo, g, fora, cfg$limiar)), 3))

# Avaliacao na colecao residual realista (somente Dr_v vistos)
novo_v <- ranking_vetor(rocchio(q0, Dr_v, Dnr, ix), ix)
fora_v <- c(Dr_v, Dnr)
cat("\n--- Colecao RESIDUAL REALISTA (Dr_v vistos no top-5) ---\n")
print(round(rbind(base    = residual(base,   g, fora_v, cfg$limiar),
                  rocchio = residual(novo_v, g, fora_v, cfg$limiar)), 3))

Teste de sanidade da teoria (curso): OK!
 [1] "d3" "d2" "d1" "d6" "d8" "d5" "d7" "d4"

Consulta de trabalho: q02 - porto de Santos 
Total de relevantes julgados (Dr): 18 
Relevantes vistos no top-5 base (Dr_v): 87 89 31 88 58 
Nao-relevantes vistos no top-5 base (Dnr):  

--- Colecao COMPLETA (Base vs Rocchio) ---
        P_at_k    AP RR nDCG_bin nDCG_grad
base         1 0.780  1    0.852     0.892
rocchio      1 0.947  1    0.985     0.989

--- Colecao RESIDUAL (Dr total: R = 0 -> AP NaN/NA) ---
        P_at_k AP RR nDCG_bin nDCG_grad
base         0 NA  0       NA         1
rocchio      0 NA  0       NA         1

--- Colecao RESIDUAL REALISTA (Dr_v vistos no top-5) ---
        P_at_k    AP RR nDCG_bin nDCG_grad
base         1 0.649  1    0.793     0.849
rocchio      1 0.498  1    0.697     0.781


## Módulo 8 — Pseudo-realimentação (PRF) em todas as consultas
**Quem digita:** Jorge Sugano Suzart.

Experimento sistemático de PRF no conjunto de desenvolvimento ($q_{01}..q_{08}$) e teste definitivo no conjunto de teste ($q_{09}..q_{12}$).

In [4]:
# Módulo 8: Pseudo-realimentacao e analise de drift
prf <- function(q0, k, ix, p = c(a = 1, b = 0.75, g = 0.15)) {
  top <- names(ranking_vetor(q0, ix))[1:k]
  ranking_vetor(rocchio(q0, top, character(0), ix = ix,
                        a = p[["a"]], b = p[["b"]], g = p[["g"]]), ix)
}

ap_q <- function(txt, grau, k, ix, cfg) {
  q0 <- vetor_consulta(preparar(txt, cfg), ix$vocab, ix$idf_tfidf)
  rk <- if (k == 0) ranking_vetor(q0, ix) else prf(q0, k, ix, cfg$rocchio)
  metricas(rk, grau, cfg$limiar)[["AP"]]
}

dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])
teste <- intersect(usar, nec$consulta[nec$conjunto == "teste"])

# Avaliacao em Dev
tab_dev <- sapply(dev, function(q) {
  a <- function(k) ap_q(consultas[[q]], qrels[[q]], k, ix, cfg)
  c(base = a(0), k2 = a(2), k3 = a(3), k5 = a(5))
})

cat("=== Tabela de Desenvolvimento (APs por consulta) ===\n")
print(round(tab_dev, 3))
cat("\nMAP medio por valor de k em Desenvolvimento:\n")
print(round(rowMeans(tab_dev), 3))

# Consultas com drift em k=2
cat("\nConsultas com severo Query Drift em k=2 (pioraram em relacao a base):\n")
print(colnames(tab_dev)[tab_dev["k2", ] < tab_dev["base", ]])

# Teste oficial executado uma unica vez com k=2 vs base
tab_teste <- sapply(teste, function(q) {
  a <- function(k) ap_q(consultas[[q]], qrels[[q]], k, ix, cfg)
  c(base = a(0), prf = a(2))
})
cat("\n=== Teste Oficial no Conjunto de Teste (q09..q12) com k=2 ===\n")
print(round(tab_teste, 3))
cat("\nMAP no Conjunto de Teste:\n")
print(round(rowMeans(tab_teste), 3))

=== Tabela de Desenvolvimento (APs por consulta) ===
       q01   q02   q03   q04   q05   q06   q07   q08
base 0.618 0.780 0.715 0.777 0.106 0.794 0.590 0.846
k2   0.298 0.553 0.591 0.458 0.138 0.633 0.331 0.583
k3   0.370 0.633 0.616 0.550 0.126 0.722 0.379 0.720
k5   0.395 0.693 0.667 0.635 0.115 0.736 0.405 0.765

MAP medio por valor de k em Desenvolvimento:
 base    k2    k3    k5 
0.653 0.448 0.515 0.551 

Consultas com severo Query Drift em k=2 (pioraram em relacao a base):
[1] "q01" "q02" "q03" "q04" "q06" "q07" "q08"

=== Teste Oficial no Conjunto de Teste (q09..q12) com k=2 ===
       q09   q10   q11   q12
base 0.591 0.666 0.614 0.578
prf  0.592 0.577 0.289 0.492

MAP no Conjunto de Teste:
 base   prf 
0.612 0.487 


## Módulo 9 — A decisão do grupo e o registro no motor
**Quem digita:** Gustavo Geminiano.

Decisão de engenharia fundamentada: **não adotar PRF em produção (`prf_k = 0`)** devido ao expressivo desvio semântico e perda de MAP, mantendo os parâmetros de Rocchio documentados em `config.R`.

In [5]:
# Módulo 9: Confirmacao da configuracao e funcao de busca do motor
source("config.R")
cat("Campos de realimentacao em config.R:\n")
cat("cfg$prf_k:", cfg$prf_k, "\n")
cat("cfg$rocchio:\n")
print(cfg$rocchio)

ranking_motor <- function(txt, ix, cfg) {
  q0 <- vetor_consulta(preparar(txt, cfg), ix$vocab, ix$idf_tfidf)
  if (cfg$prf_k == 0) ranking_vetor(q0, ix) else prf(q0, cfg$prf_k, ix, cfg$rocchio)
}

cat("\nTop-5 retornado pelo motor com a decisao final (q02 - porto de Santos):\n")
print(round(head(ranking_motor(consultas[["q02"]], ix, cfg), 5), 3))

Campos de realimentacao em config.R:
cfg$prf_k: 0 
cfg$rocchio: 
   a    b    g 
1.00 0.75 0.15 

Top-5 retornado pelo motor com a decisao final (q02 - porto de Santos):
   87    89    31    88    58 
0.312 0.301 0.285 0.280 0.274 


## Fechamento da Sessão e Snapshot de Estado
Persistência e anexação do estado da memória do R na Ficha do Projeto e no Consolidado de Grupo.

In [6]:
# Anexacao automatica de estado nos consolidados
if (file.exists("00_FICHA_PROJETO.md")) {
  anexar_estado("00_FICHA_PROJETO.md")
  cat("Estado anexado a 00_FICHA_PROJETO.md com sucesso!\n")
}
if (file.exists("aula06_parteD_consolidado.md")) {
  anexar_estado("aula06_parteD_consolidado.md")
  cat("Estado anexado a aula06_parteD_consolidado.md com sucesso!\n")
}
cat("Sessao da Aula 06 Parte D finalizada com 100% de sucesso!\n")

Estado anexado a 00_FICHA_PROJETO.md com sucesso!
Estado anexado a aula06_parteD_consolidado.md com sucesso!
Sessao da Aula 06 Parte D finalizada com 100% de sucesso!
